# 01 Macro Data Collection

This notebook downloads a small set of additional market factors that may help interpret cryptocurrency market behavior in later project stages.

| Macro Factor | Meaning          |
| --- |------------------|
| **DXY** | US dollar index  |
| **VIX** | Volatility index |

These series are downloaded, cleaned, aligned to the crypto dataset date range, and saved for later use in regime interpretation and possible model extensions.

In [5]:
from pathlib import Path
from typing import Dict

import pandas as pd
import yfinance as yf

## Configuration

This section defines the project paths, selected macro tickers, and output locations.

In [6]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MACRO_TICKERS: Dict[str, str] = {
    "DXY": "DX=F",
    "VIX": "^VIX",
}

START_DATE = "2017-01-01"
END_DATE = None
INTERVAL = "1d"

# Reference dataset for alignment
CRYPTO_ALIGNED_CLOSE_INPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_close_aligned.csv"

# Raw macro dataset
MACRO_RAW_OUTPUT_PATH = DATA_RAW_DIR / "macro_long_raw.csv"

# Clean macro datasets
MACRO_WIDE_FULL_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_close_full.csv"
MACRO_WIDE_ALIGNED_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_wide_close_aligned.csv"

# Macro datasets
MACRO_QUALITY_SUMMARY_OUTPUT_PATH = DATA_PROCESSED_DIR / "macro_data_quality_summary.csv"

## Download raw macro data

The following function downloads daily market data for a single macro factor and returns it in a consistent tabular structure.

In [7]:
def download_single_series(
    ticker: str,
    symbol: str,
    start_date: str,
    end_date: str | None,
    interval: str,
    ) -> pd.DataFrame:
    """
    Download daily market data for a single ticker from yfinance.

    Parameters
    ----------
    ticker : str
        Internal ticker name used in the project, e.g. DXY.
    symbol : str
        Yahoo Finance symbol.
    start_date : str
        Download start date in YYYY-MM-DD format.
    end_date : str | None
        Download end date in YYYY-MM-DD format. If None, the latest available date is used.
    interval : str
        Data interval, e.g. '1d'.

    Returns
    -------
    pd.DataFrame
        Raw market data with one row per date.
    """
    df = yf.download(
        symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        progress=False,
    )

    if df.empty:
        raise ValueError(f"No data returned for {ticker} ({symbol}).")

    df = df.reset_index()
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    available_columns = [col for col in expected_columns if col in df.columns]

    df = df[available_columns].copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["Ticker"] = ticker
    df["Symbol"] = symbol

    return df

In [8]:
raw_frames = []

for ticker, symbol in MACRO_TICKERS.items():
    df_ticker = download_single_series(
        ticker=ticker,
        symbol=symbol,
        start_date=START_DATE,
        end_date=END_DATE,
        interval=INTERVAL,
    )
    raw_frames.append(df_ticker)

macro_raw_df = pd.concat(raw_frames, ignore_index=True)
macro_raw_df = macro_raw_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

macro_raw_df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Symbol
0,2017-01-03,102.815002,103.815002,102.625000,103.200996,103.200996,36179,DXY,DX=F
1,2017-01-04,103.349998,103.419998,102.385002,102.709000,102.709000,60532,DXY,DX=F
2,2017-01-05,102.065002,102.500000,101.294998,101.528000,101.528000,60532,DXY,DX=F
3,2017-01-06,101.540001,102.300003,101.394997,102.209000,102.209000,31020,DXY,DX=F
4,2017-01-09,102.245003,102.489998,101.845001,101.908997,101.908997,33331,DXY,DX=F


## Save raw combined dataset

The raw combined dataset is stored before further transformation so the original download remains available for inspection and reproducibility.

In [9]:
macro_raw_df.to_csv(MACRO_RAW_OUTPUT_PATH, index=False)
print(f"Saved raw macro dataset to: {MACRO_RAW_OUTPUT_PATH}")

Saved raw macro dataset to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/raw/macro_long_raw.csv


## Basic inspection

In [10]:
print(macro_raw_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4630 entries, 0 to 4629
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype        
---  ------     --------------  -----        
 0   Date       4630 non-null   datetime64[s]
 1   Open       4630 non-null   float64      
 2   High       4630 non-null   float64      
 3   Low        4630 non-null   float64      
 4   Close      4630 non-null   float64      
 5   Adj Close  4630 non-null   float64      
 6   Volume     4630 non-null   int64        
 7   Ticker     4630 non-null   str          
 8   Symbol     4630 non-null   str          
dtypes: datetime64[s](1), float64(5), int64(1), str(2)
memory usage: 325.7 KB
None


In [11]:
macro_raw_df.isna().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
Ticker       0
Symbol       0
dtype: int64

In [12]:
macro_raw_df.duplicated(subset=["Ticker", "Date"]).sum()

0

## Data quality summary by factor

This summary shows the available history and missing values for each selected macro factor.

In [13]:
summary_dict = {
    "symbol": ("Symbol", "first"),
    "first_date": ("Date", "min"),
    "last_date": ("Date", "max"),
    "n_rows": ("Date", "count"),
    "missing_open": ("Open", lambda s: s.isna().sum()),
    "missing_high": ("High", lambda s: s.isna().sum()),
    "missing_low": ("Low", lambda s: s.isna().sum()),
    "missing_close": ("Close", lambda s: s.isna().sum()),
    "missing_volume": ("Volume", lambda s: s.isna().sum()),
}

if "Adj Close" in macro_raw_df.columns:
    summary_dict["missing_adj_close"] = ("Adj Close", lambda s: s.isna().sum())

macro_quality_summary = (
    macro_raw_df
    .groupby("Ticker")
    .agg(**summary_dict)
    .reset_index()
)

macro_quality_summary

,Ticker,symbol,first_date,last_date,n_rows,missing_open,missing_high,missing_low,missing_close,missing_volume,missing_adj_close
0,DXY,DX=F,2017-01-03,2026-03-16,2317,0,0,0,0,0,0
1,VIX,^VIX,2017-01-03,2026-03-17,2313,0,0,0,0,0,0


In [14]:
macro_quality_summary.to_csv(MACRO_QUALITY_SUMMARY_OUTPUT_PATH, index=False)
print(f"Saved macro quality summary to: {MACRO_QUALITY_SUMMARY_OUTPUT_PATH}")

Saved macro quality summary to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/macro_data_quality_summary.csv


## Create wide-format close dataset

For later use, the macro series are stored in wide format with one column per factor.

In [15]:
macro_wide_full_df = (
    macro_raw_df
    .pivot(index="Date", columns="Ticker", values="Close")
    .sort_index()
)

macro_wide_full_df = macro_wide_full_df.dropna(how="all")

print("Macro full shape:", macro_wide_full_df.shape)
macro_wide_full_df.head()

Macro full shape: (2318, 2)


Ticker,DXY,VIX
Date,,
2017-01-03,103.200996,12.85
2017-01-04,102.709000,11.85
2017-01-05,101.528000,11.67
2017-01-06,102.209000,11.32
2017-01-09,101.908997,11.56


## Align macro data to the crypto modeling period

The aligned macro dataset is restricted to the date range of the aligned crypto dataset so that both can be merged later without additional date filtering.

In [16]:
crypto_aligned_close_df = pd.read_csv(
    CRYPTO_ALIGNED_CLOSE_INPUT_PATH,
    parse_dates=["Date"],
    index_col="Date",
)

print("Crypto aligned start date:", crypto_aligned_close_df.index.min())
print("Crypto aligned end date:", crypto_aligned_close_df.index.max())

Crypto aligned start date: 2020-04-10 00:00:00
Crypto aligned end date: 2026-03-16 00:00:00


In [17]:
macro_wide_aligned_df = macro_wide_full_df.loc[
    crypto_aligned_close_df.index.min():crypto_aligned_close_df.index.max()
].copy()

print("Macro aligned shape before reindex:", macro_wide_aligned_df.shape)

Macro aligned shape before reindex: (1494, 2)


## Reindex to the crypto aligned dates

The macro series may not naturally have exactly the same date index as the crypto dataset. To ensure compatibility, the macro data is reindexed to the aligned crypto dates.

In [18]:
macro_wide_aligned_df = macro_wide_aligned_df.reindex(crypto_aligned_close_df.index)

print("Macro aligned shape after reindex:", macro_wide_aligned_df.shape)
macro_wide_aligned_df.head()

Macro aligned shape after reindex: (2167, 2)


Ticker,DXY,VIX
Date,,
2020-04-10,NaN,NaN
2020-04-11,NaN,NaN
2020-04-12,NaN,NaN
2020-04-13,99.335999,41.169998
2020-04-14,98.885002,37.759998


## Missing values after alignment

Some missing values may appear because macro series and crypto series do not always share exactly the same calendar. This is expected and should be documented before later use.

In [19]:
print("Missing values in macro full dataset:")
print(macro_wide_full_df.isna().sum())

print()
print("Missing values in macro aligned dataset:")
print(macro_wide_aligned_df.isna().sum())

Missing values in macro full dataset:
Ticker
DXY    1
VIX    5
dtype: int64

Missing values in macro aligned dataset:
Ticker
DXY    673
VIX    678
dtype: int64


<div class="alert alert-block alert-info">
These missing values mainly occur on weekends and market holidays, since macro factors follow trading calendars while cryptocurrencies trade continuously.
</div>

## Save cleaned outputs

Two versions are stored:

- full macro dataset with the complete historical range
- aligned macro dataset restricted to the crypto modeling period

In [20]:
macro_wide_full_df.to_csv(MACRO_WIDE_FULL_OUTPUT_PATH, index=True)
macro_wide_aligned_df.to_csv(MACRO_WIDE_ALIGNED_OUTPUT_PATH, index=True)

print(f"Saved macro full dataset to: {MACRO_WIDE_FULL_OUTPUT_PATH}")
print(f"Saved macro aligned dataset to: {MACRO_WIDE_ALIGNED_OUTPUT_PATH}")

Saved macro full dataset to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/macro_wide_close_full.csv
Saved macro aligned dataset to: /home/theodora/PycharmProjects/HSLU_FS25_DSPRO2/data/processed/macro_wide_close_aligned.csv
